# A critic-actor like architecture for creating a human-like chess engine that adapts on real time to play at the same level as its opponent.
## Overview
This project aims to develop a chess engine actor that receives a board state and an elo and then plays like a human with that elo would do.
The critic observes the entire game movements history and predicts which elo each player is playing at based on their previous moves.
A PID controller is then used to adjust the actor's elo on the next move so that the critic's prediction for the actor's elo matches the player's elo in the long run.

## Components
1. **Chess Engine Actor**:
   - Input: current board state, target elo.
   - Output: Next move as a probability distribution over legal moves.
2. **Critic**:
   - Input: Full game history as a sequence of fen strings and uci move pairs.
   - Output: Predicted elo for both players.
3. **PID Controller**:
   - Input: Target elo for the actor, predicted elo from the critic.
   - Output: Adjusted elo for the actor for the next move.
## Tools and Libraries
- Python-Chess: For handling chess logic and board states.
- TensorFlow/PyTorch: For building and training the critic model.
- Pandas/Numpy: For loading the training data and performing data manipulation.
- Scikit-learn: For additional machine learning utilities.
## Data
### Datasets
The lichess database is used as the main source of training data. It contains ober a billion games played by human and bot players alike. The databse constains tags with the elo of each player, player type (human or bot), date of the game, winner, etc.
# Training
## Actor Training
The chess engine actor is trained using supervised learning on a dataset of chess games with human players of various elos.
A random game and game position is sampled from the dataset, the position fen and the player's elo are extracted, and the actor is trained to predict the next move made by the player.
The loss function used is the cross-entropy loss between the predicted move probabilities and the actual move made by the player.
## Critic Training
The critic is trained using supervised learning on the same dataset of chess games.
A random game is sampled from the dataset, and partial game histories as a sequence of position fen strings and uci move pairs along with the known elos of both players are extracted.
The critic is trained to predict the elos of both players based on the game history.
The loss function used is the mean squared error between the predicted elos and the actual elos of the players.
## PID Controller Tuning
The PID controller parameters (Kp, Ki, Kd) are tuned using a grid search approach on a validation set of games.
The performance metric used for tuning is the mean absolute error between the target elo and the predicted elo from the critic over a series of moves.

# Inference
During inference, the chess engine actor receives the current board state and the target elo it should play at.
The critic observes the full game history as a sequence of position fen strings and uci move pairs and predicts the elos of both players.
The PID controller then adjusts the actor's elo for the next move based on the difference between the target elo and the predicted elo from the critic.
This process continues for each move in the game, allowing the chess engine actor to adapt its playing style in real-time to match the skill level of its opponent.
# Demo
A demo interface is created to allow users to play against the chess engine actor.
A simple python UI is built using Tkinter where users can play against the chess engine actor and see the estimated elo of both players in real-time as the game progresses.

In [2]:
!pip install chess
!pip install torch
!pip install pandas
!pip install matplotlib
!pip install scikit-learn
# import necessary libraries
import chess
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 84.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=b72745e2a03d2a8e29783e7e46c6d264938ed79a6fe38a71c517e1f5c52535cf
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


# Piece position encoding
## Overview
To effectively train the chess engine actor we need to encode the chess board positions in a more explicit way than the fen string. We use Chess.Board to recreate the position and then treat the pieces as tokens in a sequence of max lenght 32 and vocabulary (pawn, tower, knight, bishop, queen, king) * (white, black)= 12 tokens plus a BOS token, the information about the piece position is then encoded using sinusoidal positional encodings and the two are summed together to create the final input embedding.

Additionall global information like castling rights, en passant square, halfmove clock and fullmove number are encoded using learned embeddings and added to the input embedding as well.

In [3]:
def encode_position(fen):
    board = chess.Board(fen)
    piece_to_token = {
        'P': 1, 'N': 2, 'B': 3, 'R': 4, 'Q': 5, 'K': 6,
        'p': 7, 'n': 8, 'b': 9, 'r': 10, 'q': 11, 'k': 12
    }
    sequence = []
    # Add BOS token (We need to allways have at least one null token like the BOS to ensure the attention mechanism has an attention sink when the board is full).
    sequence.append([0,[0,0]])
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            token = piece_to_token[piece.symbol()]
            file, rank = chess.square_file(square), chess.square_rank(square)
            sequence.append([token, [file, rank]])
    # Pad sequence to max length 32 + BOS
    while len(sequence) < 33:
        sequence.append([0,[0,0]])
    sequence = sequence[:33]
    sequence = np.array(sequence)
    tokens = sequence[:,0]
    positions = sequence[:,1]
    # Create positional encodings
    d_model = 64
    pos_enc = np.zeros((len(positions), d_model))
    for pos_idx, (file, rank) in enumerate(positions):
        for i in range(d_model // 2):
            angle = (file + rank) / (10000 ** (2 * i / d_model))
            pos_enc[pos_idx, 2 * i] = math.sin(angle)
            pos_enc[pos_idx, 2 * i + 1] = math.cos(angle)
    # Create token embeddings
    vocab_size = 13  # 12 pieces + padding/BOS token
    token_emb = np.random.randn(vocab_size, d_model) * 0.01
    token_embeddings = token_emb[tokens]
    #create the global features embeddings and add them to the final embedding
    castling_rights = board.castling_rights
    en_passant_square = board.ep_square if board.ep_square is not None else 64
    halfmove_clock = board.halfmove_clock
    fullmove_number = board.fullmove_number
    global_features = np.array([castling_rights, en_passant_square, halfmove_clock, fullmove_number])
    global_emb = np.random.randn(4, d_model) * 0.01
    global_embedding = np.sum(global_emb, axis=0)
    global_embedding = np.tile(global_embedding, (33, 1))
    # Sum token embeddings, positional encodings and global embeddings
    final_embedding = token_embeddings + pos_enc + global_embedding
    return final_embedding # Shape (33, d_model)


# Movement encoding
## Overview
The total number of possible valid chess moves is under 2000. We create a vocabulary of all possible moves and request the model to output a logit of size VOCAB_SIZE for each position, the logit is then masked using chess.Board.legal_moves to filter out invalid moves and finally a softmax is applied to get the move probabilities.
## valid moves vocabulary creation
The vocabulary of valid moves is defined based on the starting position and possible ending positions for each piece type, then crown promotions and castling moves are added to the vocabulary.
The vocabulary is created as follows:
### Diagonal moves
- Bishop, Queen, pawn captures, etc: All possible diagonal moves from each square on the board 64 starts * max 7 squares = 448 moves
### Straight moves
- Rook, Queen, pawn forward moves, etc: All possible straight moves from each square on the board 64 starts * max 7 squares = 448 moves
### Knight moves
- All possible knight moves from each square on the board (16 starts have 8 possible moves (center squares), 16 starts have 6 possible moves (edge squares), 4 starts have 4 possible moves, etc)
### Crown promotions
- All possible movement * promotion piece combinations for pawns reaching the last rank (8 files * 4 promotion pieces * 2 colors * 3 possible ending squares) - ending squares out of board
### Castling moves
- White king side, white queen side, black king side, black queen side

In [4]:
def create_vocabulary_dict():
    vocab_dict = {}
    index = 0
    oldindex = 0
    #Null move
    vocab_dict["0000"] = index
    index += 1
    print("{} null move added on index {} for padding purposes.".format(index - oldindex, oldindex))
    oldindex = index
    # Diagonal moves
    for start_square in chess.SQUARES:
        for file_offset, rank_offset in [(1, 1), (-1, 1), (1, -1), (-1, -1)]:
            for distance in range(1, 8):
                file = chess.square_file(start_square) + file_offset * distance
                rank = chess.square_rank(start_square) + rank_offset * distance
                if 0 <= file < 8 and 0 <= rank < 8:  # Ensure the move stays within the board
                    end_square = chess.square(file, rank)
                    move_uci = chess.Move(start_square, end_square).uci()
                    vocab_dict[move_uci] = index
                    index += 1
                else:
                    break
    print("{} diagonal moves added on indexes {} to {}:".format(index - oldindex, oldindex, index-1))
    oldindex = index
    # Straight moves
    for start_square in chess.SQUARES:
        for file_offset, rank_offset in [(0, 1), (0, -1), (1, 0), (-1, 0)]:  # Vertical and horizontal directions
            for distance in range(1, 8):
                file = chess.square_file(start_square) + file_offset * distance
                rank = chess.square_rank(start_square) + rank_offset * distance
                if 0 <= file < 8 and 0 <= rank < 8:  # Ensure the move stays within the board
                    end_square = chess.square(file, rank)
                    move_uci = chess.Move(start_square, end_square).uci()
                    vocab_dict[move_uci] = index
                    index += 1
                else:
                    break
    print("{} straight moves added on indexes {} to {}:".format(index - oldindex, oldindex, index-1))
    oldindex = index
    # Knight moves
    knight_moves = [(2, 1), (2, -1), (-2, 1), (-2, -1), (1, 2), (1, -2), (-1, 2), (-1, -2)]
    for file in range(8):
        for rank in range(8):
            start_square = chess.square(file, rank)
            for move in knight_moves:
                end_file = file + move[0]
                end_rank = rank + move[1]
                if 0 <= end_file < 8 and 0 <= end_rank < 8:  # Ensure the move stays within the board
                    end_square = chess.square(end_file, end_rank)
                    move_uci = chess.Move(start_square, end_square).uci()
                    vocab_dict[move_uci] = index
                    index += 1
    print("{} knight moves added:".format(index - oldindex))
    oldindex = index
    # Crown promotions
    promotionPieces = [chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT]
    for file in range(8):
        for piece in promotionPieces:
            for color in [chess.WHITE, chess.BLACK]:
                startRank = 6 if color == chess.WHITE else 1
                endRank = 7 if color == chess.WHITE else 0
                startSquare = chess.square(file, startRank)

                # Forward promotion
                endSquare = chess.square(file, endRank)
                moveUci = chess.Move(startSquare, endSquare, promotion=piece).uci()
                vocab_dict[moveUci] = index
                index += 1

                # Capture promotions
                for offset in [-1, 1]:  # Diagonal captures
                    captureFile = file + offset
                    if 0 <= captureFile < 8:  # Ensure the capture stays within the board
                        endSquare = chess.square(captureFile, endRank)
                        moveUci = chess.Move(startSquare, endSquare, promotion=piece).uci()
                        vocab_dict[moveUci] = index
                        index += 1
    print("{} crown promotion moves added:".format(index - oldindex))
    oldindex = index
    # Castling moves
    castling_moves = ["e1g1", "e1c1", "e8g8", "e8c8"]
    for move in castling_moves:
        vocab_dict[move] = index
        index += 1
    print("{} castling moves added:".format(index - oldindex))
    return vocab_dict

vocabulary_dict = create_vocabulary_dict()

1 null move added on index 0 for padding purposes.
560 diagonal moves added on indexes 1 to 560:
896 straight moves added on indexes 561 to 1456:
336 knight moves added:
176 crown promotion moves added:
4 castling moves added:


## Masking invalid moves
The actor requires a mask of legal moves to filter out invalid moves from the output logits. This is done by creating a Chess.Board object from the fen string and getting the legal moves to create a mask.

In [5]:
def get_legal_moves_mask(fen):
    board = chess.Board(fen)
    legal_moves = list(board.legal_moves)
    mask = np.zeros(len(vocabulary_dict), dtype=np.float32)
    for move in legal_moves:
        move_uci = move.uci()
        if move_uci in vocabulary_dict:
            mask[vocabulary_dict[move_uci]] = 1.0
    return mask

# Sequence encoding with RoPE
## Overview
To encode the sequence of moves in the game history, we use Rotary Positional Encoding (RoPE) which allows the model to capture the relative positions of moves in the sequence effectively.
## Implementation
The RoPE is implemented as follows:
1. For each token in the sequence, compute its positional encoding using sine and cosine functions based on its position index.
2. Apply the RoPE transformation to the token embeddings by rotating them in the embedding space according to their positional encodings.
3. Sum the RoPE-transformed embeddings with the original token embeddings to create the final input embeddings for the model.

In [6]:
class RoPEPositionEncoder(torch.nn.Module): # Rotary Positional Encoding to be used in the transformer
    def __init__(self, vocab_size: int, emb_dim: int, base: int = 10000):
        """
        vocab_size: size of the vocabulary, if 0 no embedding layer is used and the input is assumed to be already embedded
        emb_dim: dimension of the embeddings. If no embedding layer is used, this must match the input dimension
        base: base frequency for RoPE
        """
        super().__init__()
        self.doEmbebding = vocab_size
        if(self.doEmbebding):
            self.emb = torch.nn.Embedding(vocab_size, emb_dim)
        self.emb_dim = emb_dim
        self.base = base

        # Precompute rotary frequencies
        half_dim = emb_dim // 2
        freq_seq = torch.arange(half_dim, dtype=torch.float32)
        freq = (self.base ** (-2 * freq_seq / emb_dim))
        self.register_buffer("freq", freq)  # [half_dim]

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """
        tokens: [seq_len, dim] or [batch, seq_len, dim]
        returns: rotated embeddings with RoPE applied
        """
        if(self.doEmbebding):
            x = self.emb(tokens)  # [seq, emb]
        else:
            x = tokens
        if x.dim() == 2:
            x = x.unsqueeze(0)  # -> [1, seq, emb]
        seq_len = x.size(1)
        # Compute position indices
        position_ids = torch.arange(seq_len, dtype=torch.float32, device=x.device)  # [seq]
        # Compute angles
        angles = position_ids[:, None] * self.freq[None, :]  # [seq, half_dim]
        # Compute sin and cos
        sin_angles = torch.sin(angles)  # [seq, half_dim]
        cos_angles = torch.cos(angles)  # [seq, half_dim]
        # Expand sin and cos to match embedding dimension
        sin_angles = torch.cat([sin_angles, sin_angles], dim=-1)  # [seq, emb]
        cos_angles = torch.cat([cos_angles, cos_angles], dim=-1)  # [seq, emb]
        # Apply RoPE
        x1, x2 = x[..., ::2], x[..., 1::2]  # Split even and odd dimensions
        x_rotated = torch.cat([x1 * cos_angles - x2 * sin_angles,
                               x1 * sin_angles + x2 * cos_angles], dim=-1)
        if x_rotated.size(0) == 1:
            x_rotated = x_rotated.squeeze(0)  # -> [seq, emb]
        return x_rotated # Shape: [seq_len, emb] or [batch, seq_len, emb]


# Actor Model Architecture
## Overview
The actor model is based on the ViT architecture instead pixel patches it stores piece tokens, the positional encodings are used in a similar way to ViT to give the model information about the piece position on the board. The model outputs a logit for each possible move in the vocabulary.
## Model Layers
1. Receives the piece position encoding as input plus a mask with the legal moves.
2. Passes the encoded position through a series of transformer encoder layers to capture the relationships between pieces on the board.
3. A final linear layer maps the output of the transformer to a logit vector of size VOCAB_SIZE representing the scores for each possible move.
4. Applies the legal moves mask to the logits to filter out invalid moves.
5. Applies a softmax function to the masked logits to obtain the probabilities for each valid move.

In [7]:
class ChessActorModel(torch.nn.Module):
    def __init__(self, d_model, nhead, num_layers):
        super(ChessActorModel, self).__init__()
        self.transformer = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead),
            num_layers=num_layers
        )
        self.fc_out = torch.nn.Linear(d_model, len(vocabulary_dict))

    def forward(self, position_encoding : torch.Tensor, legal_moves_mask: torch.Tensor) -> torch.Tensor:
        x = self.transformer(position_encoding) # [seq_len, batch_size, d_model]
        logits = self.fc_out(x)
        # Apply legal moves mask
        masked_logits = logits.masked_fill(~legal_moves_mask, float('-inf'))
        probabilities = torch.nn.functional.softmax(masked_logits, dim=-1)
        return probabilities

# Critic Model Architecture
## Overview
The critic model is based on a transformer architecture receiving the full game history as a sequence of positionEncodings and UCImove index pairs, the model outputs the predicted elo for both players.
## Model Layers
1. Uses the RoPE to encode the sequence of positionEncodings and UCImove index pairs.
2. Passes the encoded sequence through a series of transformer encoder layers to capture the relationships between moves in the game history.
3. A final linear layer maps the output of the transformer to a vector of size 2 representing the predicted elos for both players.

In [8]:
class ChessCriticModel(torch.nn.Module):
    def __init__(self, d_model, nhead, num_layers):
        super(ChessCriticModel, self).__init__()
        self.transformer = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead),
            num_layers=num_layers
        )
        self.fc_out = torch.nn.Linear(d_model, 2)  # Output two scalar values for elos

    def forward(self, gameHistoryPositions, gameHistoryMoves):
        """
        gameHistoryPositions: tensor of shape (batchSize, seqLen, positionEmbeddingDim)
        gameHistoryMoves: tensor of shape (batchSize, seqLen) with move indices
        """
        # Encode moves using RoPE
        rope_encoder = RoPEPositionEncoder(len(vocabulary_dict), d_model).to(device)
        move_embeddings = rope_encoder(gameHistoryMoves.to(device))  # Shape: (batchSize, seqLen, d_model)

        # Combine position embeddings and move embeddings
        x = gameHistoryPositions.to(device) + move_embeddings  # Shape: (batchSize, seqLen, d_model)

        # Pass through the transformer
        x = self.transformer(x)  # Shape: (batchSize, seqLen, d_model)
        x = x.mean(dim=1)  # Aggregate over sequence length: Shape: (batchSize, d_model)
        elos = self.fc_out(x)  # Shape: (batchSize, 2)

        return elos


# Dataset and DataLoaders
## Overview
The dataset contains information about chess games, including player ratings, game outcomes, and other relevant features. This data will be used to analyze player performance and calculate Elo ratings.
2 different DataLoaders are created, one for the actor and one for the critic, the actor DataLoader samples random positions from random games along with the player's elo and the next move made.


In [14]:
import os
import io
import gzip
from torch.utils.data import Dataset, DataLoader
import requests
import chess.pgn
import random

# ---------------------------------------Actor DataLoader ----------------------------------------------

class LichessActorDataset(Dataset):
    """
    Streams PGN games from Lichess monthly dumps.
    Produces (positions, masks) for each game, where:
      positions:  [seq_len, 33, 64]
      masks:      [seq_len, vocab_size]
    """

    def __init__(self, pgn_paths):
        """
        pgn_paths : list[str]
            Paths to .pgn.gz Lichess database files.
        """
        self.pgn_paths = pgn_paths
        self.game_offsets = []

        # Precompute byte offsets for games so __getitem__ is O(1)
        for path in pgn_paths:
            self._index_pgn_file(path)

    def _index_pgn_file(self, path):
        """Store file offsets of each game for fast random access."""
        print(f"Indexing {path}...")
        with gzip.open(path, "rt", encoding="utf-8", errors="ignore") as f:
            offset = f.tell()
            line = f.readline()

            while line:
                # Each game begins with a PGN tag
                if line.startswith("[Event "):
                    self.game_offsets.append((path, offset))
                offset = f.tell()
                line = f.readline()

        print(f"Indexed {len(self.game_offsets)} games so far.")

    def __len__(self):
        return len(self.game_offsets)

    def _load_game(self, path, offset):
        """Load one PGN game using precomputed offset."""
        with gzip.open(path, "rt", encoding="utf-8", errors="ignore") as f:
            f.seek(offset)
            game = chess.pgn.read_game(f)
        return game

    def _is_human_game(self, game):
        """Reject engine games or variants."""
        headers = game.headers

        # Reject known variants
        variant = headers.get("Variant", "").lower()
        if variant not in ("standard", ""):
            return False

        # Reject known engine generated games
        if "Engine" in headers.get("White", "") or "Engine" in headers.get("Black", ""):
            return False

        # Lichess adds tags like: BlackIsBot, WhiteIsBot
        if headers.get("WhiteIsAI") == "true":
            return False
        if headers.get("BlackIsAI") == "true":
            return False

        return True

    def __getitem__(self, idx):
        path, offset = self.game_offsets[idx]
        game = self._load_game(path, offset)

        if not self._is_human_game(game):
            # Fallback: recursively sample another game
            return self[(idx + 1) % len(self)]

        board = game.board()

        positions = []
        masks = []

        # Initial position
        fen0 = board.fen()
        positions.append(encode_position(fen0))                 # shape: (33, 64)
        masks.append(get_legal_moves_mask(fen0))               # shape: (vocab_size)

        # Loop through moves
        for move in game.mainline_moves():
            board.push(move)
            fen = board.fen()

            positions.append(encode_position(fen))
            masks.append(get_legal_moves_mask(fen))

        # Stack → (seq_len, 33, 64) / (seq_len, vocab)
        pos_tensor = torch.stack(positions, dim=0)
        mask_tensor = torch.stack(masks, dim=0)

        return pos_tensor, mask_tensor


# --- Collate function for variable-length games ---
def actor_collate_fn(batch):
    """
    Pads sequences to max game length in batch.
    """
    positions_list = [b[0] for b in batch]
    masks_list = [b[1] for b in batch]

    batch_size = len(batch)
    seq_lens = [p.shape[0] for p in positions_list]
    max_len = max(seq_lens)

    dmodel = positions_list[0].shape[1:]
    vocab_size = masks_list[0].shape[1]

    # Allocate padded tensors
    positions_padded = torch.zeros(batch_size, max_len, *dmodel)
    masks_padded = torch.zeros(batch_size, max_len, vocab_size)
    attention_mask = torch.zeros(batch_size, max_len, dtype=torch.bool)

    for i, (pos, mask) in enumerate(zip(positions_list, masks_list)):
        L = pos.shape[0]
        positions_padded[i, :L] = pos
        masks_padded[i, :L] = mask
        attention_mask[i, :L] = True

    return positions_padded, masks_padded, attention_mask


# --- Convenience loader ---
def make_lichess_actor_dataloader(pgn_paths, batch_size=8, num_workers=2, shuffle=True, split = 0.8):
    dataset = LichessActorDataset(pgn_paths)
    total_size = len(dataset)
    train_size = int(total_size * split)
    val_size = total_size - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=shuffle)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False)

    return train_loader, val_loader

# ---------------------------------------Critic DataLoader ----------------------------------------------

pos_rope = RoPEPositionEncoder(vocab_size=0, emb_dim=2112)
move_rope = RoPEPositionEncoder(vocab_size=len(vocabulary_dict), emb_dim=1)

class LichessCriticDataset(Dataset):
    """
    Loads human chess games from Lichess PGN dumps.
    For each item, returns a truncated partial game and produces
    two sequences:
      - position embeddings via encode_position + RoPE
      - move token embeddings via vocabulary_dict + RoPE
    """

    def __init__(
        self,
        pgn_paths,
        vocabulary_dict,
        pos_rope,
        move_rope,
        max_seq_len=200,
        pad_token=0
    ):
        self.pgn_paths = pgn_paths
        self.vocab = vocabulary_dict
        self.pos_rope = pos_rope
        self.move_rope = move_rope
        self.max_seq_len = max_seq_len
        self.pad_token = pad_token

        self.game_offsets = []
        for path in pgn_paths:
            self._index_pgn_file(path)

    # ----------------- Indexing ------------------

    def _index_pgn_file(self, path):
        print(f"Indexing {path}...")
        with gzip.open(path, "rt", encoding="utf-8", errors="ignore") as f:
            offset = f.tell()
            line = f.readline()

            while line:
                if line.startswith("[Event "):
                    self.game_offsets.append((path, offset))
                offset = f.tell()
                line = f.readline()

        print(f"Total indexed games: {len(self.game_offsets)}")

    def __len__(self):
        return len(self.game_offsets)

    def _load_game(self, path, offset):
        with gzip.open(path, "rt", encoding="utf-8", errors="ignore") as f:
            f.seek(offset)
            return chess.pgn.read_game(f)

    def _is_human_game(self, game):
        headers = game.headers

        # Reject variants
        variant = headers.get("Variant", "").lower()
        if variant not in ("standard", ""):
            return False

        # Reject bots
        if headers.get("WhiteIsAI") == "true":
            return False
        if headers.get("BlackIsAI") == "true":
            return False

        # Reject games marked with engines
        if "Engine" in headers.get("White", ""):
            return False
        if "Engine" in headers.get("Black", ""):
            return False

        return True

    # ----------------- Main logic ------------------

    def __getitem__(self, idx):
        path, offset = self.game_offsets[idx]
        game = self._load_game(path, offset)

        # fallback if invalid game
        if not self._is_human_game(game):
            return self[(idx + 1) % len(self)]

        board = game.board()

        # Extract full game sequence
        positions = []
        moves = []

        # First position
        positions.append(board.fen())
        for mv in game.mainline_moves():
            moves.append(mv)
            board.push(mv)
            positions.append(board.fen())

        # truncate game at some random point
        seq_len = len(moves)
        if seq_len == 0:
            return self[(idx + 1) % len(self)]

        k = random.randint(1, seq_len)   # 1..seq_len inclusive

        trunc_positions = positions[:k]     # k positions
        trunc_moves = moves[:k]             # k moves

        # --- Convert positions using encode_position ---
        pos_tensors = []
        for fen in trunc_positions:
            pos = encode_position(fen)     # (33,64)
            pos = pos.reshape(-1)          # flatten to (2112)
            pos_tensors.append(pos)

        pos_tensors = torch.stack(pos_tensors, dim=0)  # (k, 2112)

        # --- Apply RoPE to positions ---
        pos_rope_emb = self.pos_rope(pos_tensors)      # (k, 2112)

        # --- Convert moves to integer tokens ---
        move_ids = []
        for mv in trunc_moves:
            uci = mv.uci()
            tok = self.vocab.get(uci, self.pad_token)
            move_ids.append(tok)

        move_ids = torch.tensor(move_ids, dtype=torch.long)   # (k,)

        # --- Apply RoPE to move tokens ---
        move_rope_emb = self.move_rope(move_ids)             # (k, move_emb_dim)

        return pos_rope_emb, move_rope_emb, k
def critic_collate_fn(batch):
    """
    Pads:
      positions → (B, T, pos_dim)
      moves → (B, T, move_emb_dim)
      attention_mask → (B, T)
    """

    batch_positions = [b[0] for b in batch]
    batch_moves = [b[1] for b in batch]
    lengths = [b[2] for b in batch]

    B = len(batch)
    T = max(lengths)   # or use fixed max_seq_len if desired

    pos_dim = batch_positions[0].shape[1]
    move_dim = batch_moves[0].shape[1]

    pos_pad = torch.zeros(B, T, pos_dim)
    move_pad = torch.zeros(B, T, move_dim)
    attn = torch.zeros(B, T, dtype=torch.bool)

    for i, (pos, mov, L) in enumerate(zip(batch_positions, batch_moves, lengths)):
        pos_pad[i, :L] = pos
        move_pad[i, :L] = mov
        attn[i, :L] = True

    return pos_pad, move_pad, attn
def make_lichess_critic_dataloader(
    pgn_paths,
    vocabulary_dict,
    pos_rope,
    move_rope,
    batch_size=8,
    num_workers=2,
    max_seq_len=200,
    pad_token=0,
    shuffle=True,
    split = 0.8,
):
    dataset = LichessCriticDataset(
        pgn_paths,
        vocabulary_dict,
        pos_rope,
        move_rope,
        max_seq_len=max_seq_len,
        pad_token=pad_token,
    )
    total_size = len(dataset)
    train_size = int(total_size * split)
    val_size = total_size - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=critic_collate_fn,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=critic_collate_fn,
        pin_memory=True,
    )
    return train_loader, val_loader

# Training


In [23]:
#Download PGN files from Lichess
if(not os.path.exists("lichess_data")):
  os.mkdir("lichess_data")
#!wget -P lichess_data https://database.lichess.org/standard/lichess_db_standard_rated_2025-11.pgn.zst #download November 2025 file
#!wget -P lichess_data https://database.lichess.org/standard/lichess_db_standard_rated_2025-10.pgn.zst #download October 2025 file
#!wget -P lichess_data https://database.lichess.org/standard/lichess_db_standard_rated_2025-09.pgn.zst #download September 2025 file

#decompress zst files to pgn
import zstandard as zstd
def decompress_zst_to_pgn(zst_path, pgn_path):
    with open(zst_path, 'rb') as compressed_file:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(compressed_file) as reader, open(pgn_path, 'wb') as out_file:
            while True:
                chunk = reader.read(16384)
                if not chunk:
                    break
                out_file.write(chunk)
pgn_paths = []
# Decompress all the zst files in the folder
for file in os.listdir("lichess_data"):
    if not file.endswith('.zst'):
        continue
    zst_path = os.path.join("lichess_data", file)
    pgn_path = zst_path.replace('.pgn.zst', '.pgn')
    decompress_zst_to_pgn(zst_path, pgn_path)

    pgn_paths.append(pgn_path)

OSError: [Errno 28] No space left on device

In [21]:
!pip install comet_ml
from google.colab import userdata
from comet_ml import start
from comet_ml.integration.pytorch import log_model
from torch.amp import autocast, GradScaler
def train_actor_model(configs: dict, experiment: start = None):
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Model, optimizer, loss function setup
    model = ChessActorModel(
        d_model=configs['d_model'],
        nhead=configs['nhead'],
        num_layers=configs['num_layers']
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=configs['learning_rate'])
    criterion = torch.nn.CrossEntropyLoss()
    scaler = GradScaler("cuda" if torch.cuda.is_available() else "cpu")
    #look for any checkpoint to resume from
    start_epoch = 0
    if os.path.exists("actor_checkpoint.pth"):
        checkpoint = torch.load("actor_checkpoint.pth")
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resumed training from epoch {start_epoch}")
    # Get data loaders
    train_loader, val_loader = make_lichess_actor_dataloader(
        pgn_paths=configs['pgn_paths'],
        batch_size=configs['batch_size'],
        num_workers=configs['num_workers'],
        shuffle=False,#This must be false to avoid the train and val sets being  when resuming from a checkpoint
        split=configs['split']
    )
    #log hyperparameters
    if experiment:
      experiment.log_parameters({
          "d_model": configs['d_model'],
          "nhead": configs['nhead'],
          "num_layers": configs['num_layers'],
          "learning_rate": configs['learning_rate'],
          "batch_size": configs['batch_size'],
          "num_epochs": configs['num_epochs'],
      })
    # Training loop
    for epoch in range(start_epoch, configs['num_epochs']):
        model.train()
        total_loss = 0.0
        for batch_idx, (positions, masks, attention_mask) in enumerate(train_loader):
            positions = positions.to(device)
            masks = masks.to(device)
            attention_mask = attention_mask.to(device)

            optimizer.zero_grad()
            outputs = model(positions, masks)
            # Shift targets to align with outputs
            targets = torch.argmax(masks, dim=-1)
            loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{configs['num_epochs']}], Loss: {avg_loss:.4f}")
        if(experiment):
          experiment.log_metric("train_loss", avg_loss, step=epoch)

        # Validation loop
        model.eval()
        val_loss = 0.0
        with autocast("cuda" if torch.cuda.is_available() else "cpu"):
          with torch.no_grad():
              for positions, masks, attention_mask in val_loader:
                  positions = positions.to(device)
                  masks = masks.to(device)
                  attention_mask = attention_mask.to(device)

                  outputs = model(positions, masks)
                  targets = torch.argmax(masks, dim=-1)
                  loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))
                  val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"Validation Loss: {avg_val_loss:.4f}")
        if experiment:
            experiment.log_metric("val_loss", avg_val_loss, step=epoch)

        # Save checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, "actor_checkpoint.pth")

# Train
#experiment = start(api_key=userdata.get('COMET_KEY'),project_name="fastelo",workspace="irsotarriva")
configs = {
    'd_model': 64,
    'nhead': 8,
    'num_layers': 4,
    'learning_rate': 1e-4,
    'batch_size': 16,
    'num_epochs': 10,
    'num_workers': 4,
    'pgn_paths': pgn_paths,
    'split': 0.8
}
train_actor_model(configs, experiment)
experiment.end()

COMET INFO: An experiment with the same configuration options is already running and will be reused.


Indexing lichess_data/lichess_db_standard_rated_2025-11.pgn.zst...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


BadGzipFile: Not a gzipped file (b'P*')